## USING STRUCTURED OUTPUTS

    Models can be requested to give outputs in a particular format, the structure can be defined and there can be a particular schema for the metadata

In [9]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Load environment variables
load_dotenv()

# Initialize Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

print("Chatbot Started! Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    response = llm.invoke([HumanMessage(content=user_input)])

    print("Bot:", response.content)

Chatbot Started! Type 'exit' to quit.

Bot: I cannot provide an exact figure for the size of the dataset I was trained on. The precise details of the training data, including its exact size, are proprietary information held by Google.

However, I can tell you that the dataset is **immense**. It comprises **trillions of tokens** (a token can be a word, part of a word, or a punctuation mark) and spans **many petabytes** of data.

This vast collection includes a diverse range of text and code from the internet, such as:
*   Books
*   Articles
*   Websites
*   Code repositories
*   Academic papers
*   And many other forms of publicly available digital text.

The goal is to provide a broad and deep understanding of human language, facts, reasoning, and various forms of expression.
Bot: That's a great question! My "memory" isn't like a human brain's memory at all. It's more accurate to think of it in a few different layers:

1.  **Long-Term "Memory" (My Training Data & Model Parameters):**
 

# Pydantic

    Field validation, descriptions and nested structures 

    -> The ... is a special Python value called Ellipsis. ("This field is required and must be provided.")

## Pydantic adds:

    validation  ""Check that the data matches the expected type and format."
    defaults
    descriptions
    constraints
    methods

    That's why most LangChain projects use Pydantic for structured outputs.

In [4]:
from pydantic import BaseModel, Field

# it is just a baseclass for creating pydantic models, and it does not enforce any specific behavior regarding required fields.
class Movi(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    genre: str = Field(..., description="The genre of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(description="The rating of the movie (0.0 to 10.0)"  )
    
    # Pydantic v2 may still treat it as required because there is no default 
    # value, but using ... makes the intent explicit and is a very 
    # common pattern.

In [6]:
model_with_struct = llm.with_structured_output(Movi)
model_with_struct.invoke([HumanMessage(content=input("Describe a movie: "))])

Movi(title='The Exorcist', year=1973, genre='Horror', director='William Friedkin', rating=8.0)

In [8]:
res = llm.invoke([HumanMessage(content="where aare you fetching the movi ratings from?")])
print(res.content)
print(res.usage_metadata)

As an AI, I don't "fetch" real-time data like a human browsing the internet or an application calling an API.

My knowledge about movie ratings comes from the vast dataset I was trained on. This dataset included a massive amount of text and information from the internet up to my last training update.

Therefore, the ratings I might cite or discuss would reflect information that was publicly available and present in my training data from various sources, such as:

*   **IMDb (Internet Movie Database)**
*   **Rotten Tomatoes** (both Tomatometer and Audience Score)
*   **Metacritic**
*   **User reviews and forums**
*   **Critical reviews from various publications**
*   **General entertainment news sites**

It's important to note that my information isn't real-time, so very recent changes to ratings or new movie releases might not be reflected. For the most up-to-date and specific ratings, you would need to check current movie review websites or databases directly.
{'input_tokens': 12, 'ou

In [11]:
# Nested structured output example

class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    age: int = Field(..., description="The age of the actor")
    role: str = Field(description="The role of the actor in the movie")
    awards: list[str] = Field(default_factory=list, description="List of awards won by the actor")
    
class MoviDetails(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    cast: list[Actor] = Field(default_factory=list, description="List of actors in the movie")
    
response = llm.invoke([HumanMessage(content="Provide details of the movie Inception including its cast.")])
print(response.content)
    

"Inception" is a critically acclaimed and commercially successful science fiction action film released in 2010, written and directed by Christopher Nolan. It is renowned for its complex narrative, stunning visual effects, and exploration of dreams and the human subconscious.

Here are the details:

## Inception (2010)

**Director & Writer:** Christopher Nolan
**Producers:** Christopher Nolan, Emma Thomas
**Music:** Hans Zimmer
**Cinematography:** Wally Pfister
**Production Companies:** Legendary Pictures, Syncopy Inc.
**Distributed by:** Warner Bros. Pictures
**Release Date:** July 16, 2010
**Runtime:** 148 minutes
**Budget:** $160 million
**Box Office:** Over $836 million worldwide

### Plot Summary:

The film centers on **Dom Cobb** (Leonardo DiCaprio), a skilled "extractor" who commits corporate espionage by entering people's dreams to steal their ideas. His unique abilities have made him a fugitive, preventing him from returning home to his children.

Cobb is offered a chance at re

# TypeDict

    no validation or constraint (just a python dictionary)

    1. TypedDict : Used to define the expected structure of a dictionary.
    2. Annotated : Adds extra metadata to a type.
    3. TypeAlias : Used to create a custom name for a type.
                    from typing_extensions import TypeAlias
                        Scores: TypeAlias = dict[str, list[int]]




In [18]:
from typing_extensions import Annotated, TypedDict

# Nested structured output example

class Actor(TypedDict):
    name: str 
    age: int 
    role: str 
    
class MoviDetails(TypedDict):
    title: str 
    year: int 
    cast: list[Actor]
    
response = llm.invoke([HumanMessage(content="Provide details of the movie Inception including its cast.")])
print(response.content)
    

"Inception" is a critically acclaimed and mind-bending science fiction action film directed by **Christopher Nolan**, released on July 16, 2010. It's renowned for its complex narrative, stunning visual effects, and philosophical themes exploring the nature of reality, dreams, and the subconscious mind.

---

### **Overview**

*   **Director:** Christopher Nolan
*   **Writers:** Christopher Nolan
*   **Producers:** Christopher Nolan, Emma Thomas
*   **Release Date:** July 16, 2010
*   **Genre:** Science Fiction, Action, Thriller, Heist Film, Psychological Drama
*   **Running Time:** 148 minutes
*   **Box Office:** Grossed over $836 million worldwide.
*   **Awards:** Won 4 Academy Awards (Best Cinematography, Best Sound Editing, Best Sound Mixing, Best Visual Effects) and was nominated for 4 others, including Best Picture and Best Original Screenplay.

---

### **Plot Summary**

The film centers on **Dom Cobb** (Leonardo DiCaprio), a skilled "extractor" who specializes in stealing valuab

# Data 